In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [9]:
load_dotenv()

True

In [10]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model='gemini-3.1-flash-lite',
    google_api_key=os.getenv('GEMINI_API_KEY')
)

In [11]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [12]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [13]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [14]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])



[{'type': 'text', 'text': 'The average distance from the Earth to the moon is about **238,855 miles** (384,400 kilometers).\n\nBecause the moon orbits the Earth in an elliptical (oval) shape rather than a perfect circle, this distance changes constantly:\n\n*   **Perigee:** When the moon is closest to Earth, it is about **225,623 miles** (363,104 km) away.\n*   **Apogee:** When the moon is farthest from Earth, it is about **252,088 miles** (405,696 km) away.\n\nTo put that distance into perspective, you could fit all the other planets in our solar system side-by-side in the space between the Earth and the moon.', 'extras': {'signature': 'EnEKbwERTTIPkSHh/Xoceiv0rPnb/Pi4HIPCjtTMcwnMgsp3HeJZgkMQhqX1JvZspmJG2s4ORIyLZZ9PpCUqckUOdqdg+cMd6ALCCkVfzdTW8GrOZTU7G21OEC6mee2je7BKcA0vLpiUa+UmSVEdmZR03A=='}}]


In [16]:
model.invoke('How far is moon from the earth?')

AIMessage(content=[{'type': 'text', 'text': 'The distance between the Earth and the Moon is not constant because the Moon follows an elliptical (oval-shaped) orbit. Here are the key numbers:\n\n*   **Average distance:** About **238,855 miles** (384,400 kilometers).\n*   **Perigee (closest approach):** About **225,623 miles** (363,104 kilometers).\n*   **Apogee (farthest distance):** About **252,088 miles** (405,696 kilometers).\n\n**To put that in perspective:**\n*   You could fit all the other planets in our solar system side-by-side in the space between the Earth and the Moon.\n*   It takes light about **1.3 seconds** to travel from the Moon to the Earth.\n*   The Apollo astronauts took about **three days** to reach the Moon.', 'extras': {'signature': 'EnEKbwERTTIPBk12u/dT0PGlWEoYej9D1VhhsJksYRsRSFeHzXK30RLtq/wj6TSvpCPGscambqxiiA6ytjBTQ+y2tAU3pwTPFD1O0X+1dL2CWLZyMoZIsCyat2o6XUXfNczBKxU5jaiZ7VBZhQJ2ghVMlw=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_